# P11 — Prédiction du Nutri-Score d'un produit alimentaire

**Contexte métier.** Plateformes de consommation (Yuka-like), acheteurs d'entreprises, nutritionnistes : prédire le Nutri-Score d'un produit à partir de sa composition pour estimer les produits sans étiquetage et détecter les formulations trompeuses. Enjeu : santé publique, réforme des recettes.

**Objectif de l'EDA.** Identifier la cible (grade Nutri-Score a-e) et les régresseurs nutritionnels (énergie, sucres, sel, fibres, protéines).

**Données.** Open Food Facts — export français (fr.openfoodfacts.org.products.csv). Licence ODbL. Base complète : 1,29 Go compressé ; l'EDA travaille sur un téléchargement partiel (60 premiers Mo = 60 000 premiers produits), suffisant pour identifier cible et régresseurs.

## 1. Téléchargement partiel (60 Mo)

L'export complet pèse 1,3 Go : on télécharge le début du fichier par requête HTTP Range (Accept-Ranges: bytes), puis on décompresse en flux en tolérant la troncature. Le même code peut télécharger plus de Mo pour entraîner ensuite.

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://static.openfoodfacts.org/data/fr.openfoodfacts.org.products.csv.gz"
TAILLE_MO = 60  # augmenter pour un echantillon plus grand
if not os.path.exists("data/fr_head.csv.gz"):
    req = urllib.request.Request(URL, headers={"Range": f"bytes=0-{TAILLE_MO * 1024 * 1024 - 1}"})
    with urllib.request.urlopen(req) as r, open("data/fr_head.csv.gz", "wb") as f:
        f.write(r.read())
print(os.path.getsize("data/fr_head.csv.gz") / 1e6, "Mo telecharges")

## 2. Extraction en flux

Décompression partielle du gz : on garde les 60 000 premières lignes du CSV (tabulations) avec les colonnes utiles, en tolérant la fin abrupte du flux.

In [ ]:
import gzip, csv
import pandas as pd, numpy as np

cols = ["code", "product_name", "brands", "categories", "pnns_groups_2", "nutriscore_grade",
        "nutriscore_score", "energy-kcal_100g", "fat_100g", "saturated-fat_100g",
        "sugars_100g", "salt_100g", "proteins_100g", "fiber_100g", "nova_group"]
rows, header = [], None
try:
    with gzip.open("data/fr_head.csv.gz", "rt", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader)
        idx = {c: header.index(c) for c in cols if c in header}
        for i, r in enumerate(reader):
            if i >= 60000:
                break
            rows.append([r[idx[c]] if c in idx else None for c in cols])
except (EOFError, OSError):
    pass  # fin de flux anticipee (fichier tronque volontairement)
df = pd.DataFrame(rows, columns=cols)
df["grade"] = df["nutriscore_grade"].replace({"unknown": np.nan, "not-applicable": np.nan})
for c in cols[6:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
print(len(df), "produits,", df["grade"].notna().sum(), "avec grade")

## 3. Analyse

Distribution des grades, moyennes nutritionnelles par grade, corrélation des nutriments avec le score Nutri-Score.

In [ ]:
dfa = df.dropna(subset=["grade"])
print("Grades :", dfa["grade"].value_counts().sort_index().to_dict())
print()
nutr = ["energy-kcal_100g", "sugars_100g", "saturated-fat_100g", "salt_100g", "proteins_100g", "fiber_100g"]
print("Moyennes par grade :")
for g in sorted(dfa["grade"].unique()):
    print(" ", g, dfa[dfa["grade"] == g][nutr].mean().round(2).to_dict())
print()
c = dfa[["nutriscore_score"] + nutr].corr("spearman")["nutriscore_score"]
print("Corr. Spearman avec le score Nutri-Score :", c.round(2).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n echantillon : 60000
- n avec grade : 24074
- grade dist : a : 4730, b : 2481, c : 3970, d : 5182, e : 7711
- pnns top10 : unknown : 9849, Dressings and sauces : 1488, Sweets : 1444, Biscuits and cakes : 1280, Cereals : 1095, Vegetables : 9…
- nova dist : 1.0 : 3079, 2.0 : 441, 3.0 : 3828, 4.0 : 14146
- nutriments stats : energy-kcal_100g : count : 3 777.00, mean : 268.52, std …, fat_100g : count : 3 777.00, mean : 13.03, std :…, saturat…
- moyennes par grade : a : energy-kcal_100g : 181.37, sugars_100…, b : energy-kcal_100g : 156.21, sugars_100…, c : energy-kcal_100g : 227.83…
- corr spearman grade : nutriscore_score : 1.00, energy-kcal_100g : 0.50, fat_100g : 0.44, saturated-fat_100g : 0.49, sugars_100g : 0.37, sal…
- nutriscore score stats : count : 24 074.00, mean : 11.20, std : 10.90, min : -17.00, 25% : 1.00, 50% : 11.00, 75% : 19.00, max : 54.00
- n marques : 8375
- missing pct : grade : 59.90, energy-kcal_100g : 80.00, sugars_100g : 81.40, salt_100g : 84.10, proteins_100g : 80.40

## 5. Variable cible et régresseurs

**Cible (classification ordinale a-e ou régression du score) : `nutriscore_grade`.** 24 074 produits gradés sur 60 000 dans l'échantillon.

**Régresseurs retenus (les nutriments pour 100 g) :**
- `energy-kcal_100g`, `sugars_100g`, `saturated-fat_100g`, `salt_100g` : corrélés positivement au score (mauvais côté)
- `proteins_100g`, `fiber_100g` : corrélés négativement (bon côté)
- `pnns_groups_2` / `categories` : la catégorie détermine la formule du score (boissons, fromages) — feature indispensable
- `nova_group` (degré de transformation) : signal complémentaire (54 % de NOVA 4 parmi les produits gradés)

**Pièges :**
- `nutriscore_score` est calculé depuis les nutriments : c'est une fuite si on prédit le grade — prédire le grade depuis les nutriments seulement reproduit la formule officielle (le projet peut être exactement ça : apprendre la formule), mais le faire savoir
- 58 % de produits sans grade : la première tâche ML peut être l'imputation
- Valeurs aberrantes (energy 18 000 kcal/100 g) : filtrage qualité indispensable
- L'échantillon head du fichier n'est pas aléatoire (produits anciens) : pour l'entraînement, échantillonner aléatoirement la base